[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/04_activation_and_gating.ipynb)

# 04. Activation and gating — activation functions vs gated FFNs

이전 GEGLU/SwiGLU section은 이미 만들어진 두 tensor를 곱하기만 해서 Transformer FFN의 실제 구조인 **두 learned input projections + gate activation + elementwise product + output projection**이 빠져 있었다.

이번 버전은 scalar activation과 gated FFN 구조를 분리해서 본다.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


device: cpu


## 1. ReLU, GELU, and SiLU are pointwise nonlinearities

이 함수들은 입력 tensor의 각 원소를 독립적으로 변환한다. gated FFN과 달리 별도 branch interaction은 없다.


In [2]:
x = torch.tensor(
    [-3.0, -1.0, 0.0, 1.0, 3.0],
    device=device,
)

print("ReLU:", F.relu(x))
print("GELU:", F.gelu(x))
print("SiLU:", F.silu(x))


ReLU: tensor([0., 0., 0., 1., 3.])
GELU: tensor([-0.0040, -0.1587,  0.0000,  0.8413,  2.9960])
SiLU: tensor([-0.1423, -0.2689,  0.0000,  0.7311,  2.8577])


## 2. Original GLU structure

GLU는 같은 input `x`에서 두 learned affine projection을 만들고 한쪽을 sigmoid gate로 사용한다. 즉 `A(x) ⊙ sigmoid(B(x))`가 핵심이다.


In [3]:
batch = torch.randn(3, 8, device=device)

value_projection = nn.Linear(8, 16).to(device)
gate_projection = nn.Linear(8, 16).to(device)

value_branch = value_projection(batch)
gate_branch = torch.sigmoid(gate_projection(batch))
glu_hidden = value_branch * gate_branch

print("value branch:", value_branch.shape)
print("gate branch:", gate_branch.shape)
print("GLU hidden:", glu_hidden.shape)


value branch: torch.Size([3, 16])
gate branch: torch.Size([3, 16])
GLU hidden: torch.Size([3, 16])


## 3. GEGLU and SwiGLU as complete Transformer FFNs

Transformer에서 쓰이는 GEGLU/SwiGLU는 gate product 뒤 다시 model dimension으로 projection한다. activation만 GELU 또는 SiLU로 달라진다.


In [4]:
class GatedFFN(nn.Module):
    def __init__(
        self,
        model_dim=8,
        hidden_dim=24,
        gate="silu",
    ):
        super().__init__()

        self.gate = gate
        self.value_projection = nn.Linear(
            model_dim,
            hidden_dim,
            bias=False,
        )
        self.gate_projection = nn.Linear(
            model_dim,
            hidden_dim,
            bias=False,
        )
        self.output_projection = nn.Linear(
            hidden_dim,
            model_dim,
            bias=False,
        )

    def forward(self, x):
        value = self.value_projection(x)
        gate_input = self.gate_projection(x)

        if self.gate == "gelu":
            gate = F.gelu(gate_input)
        elif self.gate == "silu":
            gate = F.silu(gate_input)
        else:
            raise ValueError(self.gate)

        hidden = value * gate
        return self.output_projection(hidden)


geglu_ffn = GatedFFN(gate="gelu").to(device)
swiglu_ffn = GatedFFN(gate="silu").to(device)

geglu_output = geglu_ffn(batch)
swiglu_output = swiglu_ffn(batch)

print("GEGLU output:", geglu_output.shape)
print("SwiGLU output:", swiglu_output.shape)


GEGLU output: torch.Size([3, 8])
SwiGLU output: torch.Size([3, 8])


## 4. Compare ordinary FFN and SwiGLU computation graphs


In [5]:
ordinary_ffn = nn.Sequential(
    nn.Linear(8, 24),
    nn.GELU(),
    nn.Linear(24, 8),
).to(device)

ordinary_output = ordinary_ffn(batch)
swiglu_output = swiglu_ffn(batch)

print("ordinary FFN:", ordinary_output.shape)
print("SwiGLU FFN:", swiglu_output.shape)
print("SwiGLU has two input projections before the product")


ordinary FFN: torch.Size([3, 8])
SwiGLU FFN: torch.Size([3, 8])
SwiGLU has two input projections before the product


## References and provenance

**GELU** — Hendrycks & Gimpel. pointwise smooth activation을 반영했다.

**GLU** — Dauphin et al. two affine branches와 sigmoid gating을 반영했다.

**GEGLU / SwiGLU** — Shazeer, *GLU Variants Improve Transformer*. separate value/gate projections, GELU/SiLU gate, elementwise product, output projection으로 complete gated FFN을 구성했다.
